In [6]:
!pip install tensorflow==2.10.0 scikit-learn pandas "numpy<2.0" matplotlib fastapi uvicorn
!pip install --upgrade "mistralai>=1.0.0" -q

# Install & Import Dulu Semua yang Dibutuhkan

In [3]:
import pandas as pd
import numpy as np
import re, ast, os, pickle, json, time, requests
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
from datetime import datetime
from collections import Counter

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
import keras
from keras import layers, regularizers

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, accuracy_score

print("TensorFlow:", tf.__version__)
print("GPU:", len(tf.config.list_physical_devices('GPU')) > 0)

TensorFlow: 2.10.0
GPU: True


# Load dan Pahami Dataset

In [4]:
# Load dataset dari Kaggle
df = pd.read_csv('resume_features.csv')

print(f"Total CV  : {df.shape[0]}")
print(f"Kolom     : {df.columns.tolist()}")
print(f"Kategori unik: {df['category'].nunique()}")
print()
print(df['category'].value_counts())

Total CV  : 12755
Kolom     : ['category', 'summary', 'highlights', 'experience', 'experience_years', 'education', 'certifications', 'skills', 'skills_list', 'skills_count']
Kategori unik: 44

category
HEALTH AND FITNESS           460
EDUCATION                    421
BUILDING AND CONSTRUCTION    416
FINANCE                      413
ACCOUNTANT                   412
SALES                        410
CONSULTANT                   406
BUSINESS ANALYST             385
HUMAN RESOURCES              379
PUBLIC RELATIONS             376
AVIATION                     372
DIGITAL MEDIA                366
BANKING                      360
APPAREL                      359
ADVOCATE                     339
INFORMATION TECHNOLOGY       338
DESIGNER                     315
ELECTRICAL ENGINEERING       312
OPERATIONS MANAGER           310
AUTOMOBILE                   299
MANAGEMENT                   288
ARTS                         285
MECHANICAL ENGINEER          279
SQL DEVELOPER                270
WEB DE

# Bersihkan dan Siapkan Fitur

In [5]:
def fill_nulls(df):
    df = df.copy()
    df['highlights']     = df['highlights'].fillna('no highlights')
    df['education']      = df['education'].fillna('no education information')
    df['certifications'] = df['certifications'].fillna('no certification')
    return df

df = fill_nulls(df)
print("Null setelah diisi:")
print(df.isnull().sum())

Null setelah diisi:
category            0
summary             0
highlights          0
experience          0
experience_years    0
education           0
certifications      0
skills              0
skills_list         0
skills_count        0
dtype: int64


In [6]:
def group_career_targets(category):
    cat = str(category).upper()

    # Tech
    if any(w in cat for w in ['REACT','WEB DESIGNING','SOFTWARE','JAVA','DOTNET',
                               'PYTHON DEVELOPER','TESTING','DEVOPS','PROGRAMMER']):
        return 'Software & Web Development'
    if any(w in cat for w in ['DATA SCIENCE','DATABASE','ETL','HADOOP','SQL DEVELOPER']):
        return 'Data Science & Engineering'
    if any(w in cat for w in ['INFORMATION TECHNOLOGY','NETWORK SECURITY','BLOCKCHAIN','SAP']):
        return 'IT Infrastructure & Security'

    # Business
    if any(w in cat for w in ['ACCOUNTANT','FINANCE','BANKING','TAX']):
        return 'Finance & Accounting'
    if any(w in cat for w in ['HUMAN RESOURCES']):
        return 'Human Resources'
    if any(w in cat for w in ['SALES','BPO','BUSINESS DEVELOPMENT','CLIENT']):
        return 'Sales & Business Development'
    if any(w in cat for w in ['BUSINESS ANALYST']):
        return 'Business Analyst'
    if any(w in cat for w in ['CONSULTANT']):
        return 'Management Consulting'
    if any(w in cat for w in ['OPERATIONS MANAGER','MANAGEMENT','PMO']):
        return 'Management & Operations'

    # Creative & Media
    if any(w in cat for w in ['ARTS','APPAREL','DESIGNER']):
        return 'Arts & Creative Design'
    if any(w in cat for w in ['PUBLIC RELATIONS','DIGITAL MEDIA']):
        return 'PR & Digital Media'
    if any(w in cat for w in ['ARCHITECTURE']):
        return 'Architecture & Design'

    # Professional Services
    if any(w in cat for w in ['HEALTH AND FITNESS','NURSE','DOCTOR','MEDICAL']):
        return 'Healthcare & Wellness'
    if any(w in cat for w in ['EDUCATION']):
        return 'Education & Training'
    if any(w in cat for w in ['ADVOCATE','LEGAL','LAW','ATTORNEY']):
        return 'Legal & Advocacy'

    # Engineering & Physical
    if any(w in cat for w in ['ELECTRICAL ENGINEERING','MECHANICAL ENGINEER',
                               'CIVIL ENGINEER','ENGINEERING']):
        return 'Engineering & Technical'
    if any(w in cat for w in ['BUILDING AND CONSTRUCTION']):
        return 'Construction'
    if any(w in cat for w in ['AUTOMOBILE','AVIATION','TRANSPORT','LOGISTIC']):
        return 'Transportation & Logistics'

    # Others
    if any(w in cat for w in ['FOOD AND BEVERAGES','CHEF','CULINARY']):
        return 'Food & Hospitality'
    if any(w in cat for w in ['AGRICULTURE']):
        return 'Agriculture'

    return 'Other Professionals'

df['category_grouped'] = df['category'].apply(group_career_targets)

other = df[df['category_grouped'] == 'Other Professionals']['category'].unique()
if len(other) > 0:
    print(f"⚠️  Kategori tidak terpetakan: {other}")
else:
    print("✅ Semua kategori terpetakan!")

print("\nDistribusi 20 Kelompok Karir:")
dist = df['category_grouped'].value_counts()
print(dist)
print(f"\nTotal kelompok: {dist.shape[0]}")

✅ Semua kategori terpetakan!

Distribusi 20 Kelompok Karir:
category_grouped
Software & Web Development      1581
Finance & Accounting            1185
Engineering & Technical          963
Arts & Creative Design           959
Data Science & Engineering       885
IT Infrastructure & Security     865
Management & Operations          811
PR & Digital Media               742
Transportation & Logistics       671
Sales & Business Development     589
Healthcare & Wellness            460
Education & Training             421
Construction                     416
Management Consulting            406
Business Analyst                 385
Human Resources                  379
Legal & Advocacy                 339
Architecture & Design            257
Food & Hospitality               222
Agriculture                      219
Name: count, dtype: int64

Total kelompok: 20


In [7]:
def engineer_features(df):
    df = df.copy()

    # Sertifikasi
    df['has_certification'] = df['certifications'].apply(
        lambda x: 0 if 'no certification' in str(x).lower() else 1
    )

    def count_certs(text):
        if 'no certification' in str(text).lower():
            return 0
        try:
            parsed = ast.literal_eval(str(text))
            if isinstance(parsed, list):
                return len(parsed)
        except:
            pass
        return len([x for x in str(text).split(',') if x.strip()])

    df['cert_count']        = df['certifications'].apply(count_certs)
    df['has_education']     = df['education'].apply(
        lambda x: 0 if 'no education' in str(x).lower() else 1
    )
    df['experience_years']  = df['experience_years'].clip(upper=40.0)
    df['skills_count']      = df['skills_count'].clip(upper=36)
    df['is_fresh_graduate'] = (df['experience_years'] < 2).astype(int)
    df['is_experienced']    = (df['experience_years'] >= 5).astype(int)
    df['has_highlights']    = df['highlights'].apply(
        lambda x: 0 if 'no highlights' in str(x).lower() else 1
    )
    return df

df = engineer_features(df)

num_cols = ['has_certification','cert_count','has_education',
            'experience_years','skills_count','is_fresh_graduate',
            'is_experienced','has_highlights']
print("Statistik Fitur Numerik:")
print(df[num_cols].describe())

Statistik Fitur Numerik:
       has_certification    cert_count  has_education  experience_years  \
count       12755.000000  12755.000000   12755.000000      12755.000000   
mean            0.400706      0.706233       0.760643         12.669573   
std             0.490061      1.782344       0.426708          9.792521   
min             0.000000      0.000000       0.000000          0.000000   
25%             0.000000      0.000000       1.000000          5.000000   
50%             0.000000      0.000000       1.000000         10.800000   
75%             1.000000      1.000000       1.000000         18.100000   
max             1.000000     67.000000       1.000000         40.000000   

       skills_count  is_fresh_graduate  is_experienced  has_highlights  
count  12755.000000       12755.000000    12755.000000    12755.000000  
mean      13.286319           0.110858        0.756409        0.827440  
std       10.368664           0.313969        0.429265        0.377881  
min    

In [10]:
def combine_text_features(row):
    parts = []

    # Skills: boost 7x — sinyal terkuat untuk klasifikasi
    if pd.notna(row['skills']) and str(row['skills']).strip():
        skills_text = str(row['skills'])
        for _ in range(7):
            parts.append(skills_text)

    # Summary: konteks karir
    if pd.notna(row['summary']) and str(row['summary']).strip():
        parts.append(str(row['summary']))

    # Experience: bukti pengalaman (batasi 1500 char)
    if pd.notna(row['experience']) and str(row['experience']).strip():
        parts.append(str(row['experience'])[:1500])

    # Highlights & Education: informasi tambahan
    if str(row['highlights']) != 'no highlights':
        parts.append(str(row['highlights'])[:400])
    if str(row['education']) != 'no education information':
        parts.append(str(row['education'])[:400])

    return ' '.join(parts)


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s,.\'\-+#/]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df['combined_text'] = df.apply(combine_text_features, axis=1).apply(clean_text)

avg_len = int(df['combined_text'].str.len().mean())
print(f"Rata-rata panjang teks : {avg_len} karakter")
print(f"Min / Max panjang       : {df['combined_text'].str.len().min()} / {df['combined_text'].str.len().max()}")
print("Contoh text (100 char):", df['combined_text'].iloc[0][:100])

Rata-rata panjang teks : 4018 karakter
Min / Max panjang       : 223 / 65045
Contoh text (100 char): accounting, ads, advertising, analytical skills, benefits, billing, budgeting, clients, customer ser


# Encode Label dan Split Data

In [11]:
le = LabelEncoder()
df['label']  = le.fit_transform(df['category_grouped'])
num_classes  = len(le.classes_)
print(f"Jumlah kelas: {num_classes}")
print("Kelas:", le.classes_.tolist())

# Class weights untuk menangani imbalance
class_weights_array = compute_class_weight(
    'balanced',
    classes=np.arange(num_classes),
    y=df['label'].values
)
class_weight_dict = dict(enumerate(class_weights_array))

NUM_NUMERIC_FEATURES = 8

NUMERIC_COLS = ['experience_years','skills_count','has_certification',
                'cert_count','has_education','is_fresh_graduate',
                'is_experienced','has_highlights']

X_text    = df['combined_text'].values
X_numeric = df[NUMERIC_COLS].values.astype(np.float32)
y_label   = df['label'].values

# Split: 70% train | 15% val | 15% test — stratified
X_text_train, X_text_temp, X_num_train, X_num_temp, y_train, y_temp = train_test_split(
    X_text, X_numeric, y_label,
    test_size=0.30, random_state=42, stratify=y_label
)
X_text_val, X_text_test, X_num_val, X_num_test, y_val, y_test = train_test_split(
    X_text_temp, X_num_temp, y_temp,
    test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nTrain : {len(X_text_train):>5} sampel")
print(f"Val   : {len(X_text_val):>5} sampel")
print(f"Test  : {len(X_text_test):>5} sampel")

Jumlah kelas: 20
Kelas: ['Agriculture', 'Architecture & Design', 'Arts & Creative Design', 'Business Analyst', 'Construction', 'Data Science & Engineering', 'Education & Training', 'Engineering & Technical', 'Finance & Accounting', 'Food & Hospitality', 'Healthcare & Wellness', 'Human Resources', 'IT Infrastructure & Security', 'Legal & Advocacy', 'Management & Operations', 'Management Consulting', 'PR & Digital Media', 'Sales & Business Development', 'Software & Web Development', 'Transportation & Logistics']

Train :  8928 sampel
Val   :  1913 sampel
Test  :  1914 sampel


# Tokenisasi Teks

In [12]:
MAX_TOKENS = 20000  
MAX_LEN    = 300     
EMBED_DIM  = 128     

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=MAX_LEN,
    ngrams=2,
    standardize='lower_and_strip_punctuation'
)
vectorizer.adapt(X_text_train)

vocab_size = len(vectorizer.get_vocabulary())
print(f"Ukuran kamus: {vocab_size:,} token")

Ukuran kamus: 20,000 token


# tf.data Pipeline

In [13]:
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_val_scaled   = scaler.transform(X_num_val)
X_num_test_scaled  = scaler.transform(X_num_test)

BATCH_SIZE = 64
AUTOTUNE   = tf.data.AUTOTUNE


def make_dataset(X_text, X_numeric, y_labels, is_train=False):
    X_tokens = vectorizer(X_text)
    if is_train:
        sample_weights = np.array(
            [class_weight_dict[lbl] for lbl in y_labels],
            dtype=np.float32
        )
        ds = tf.data.Dataset.from_tensor_slices((
            {'text_input': X_tokens, 'numeric_input': X_numeric},
            y_labels,
            sample_weights
        )).shuffle(buffer_size=5000, seed=42)   # ↑ buffer lebih besar
    else:
        ds = tf.data.Dataset.from_tensor_slices((
            {'text_input': X_tokens, 'numeric_input': X_numeric},
            y_labels
        ))
    return ds.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)


train_ds = make_dataset(X_text_train, X_num_train_scaled, y_train, is_train=True)
val_ds   = make_dataset(X_text_val,   X_num_val_scaled,   y_val)
test_ds  = make_dataset(X_text_test,  X_num_test_scaled,  y_test)

print(f"Train batches: {len(list(train_ds))}")
print(f"Val batches  : {len(list(val_ds))}")
print(f"Test batches : {len(list(test_ds))}")

Train batches: 140
Val batches  : 30
Test batches : 30


# Komponen Kustom Tensorflow

In [14]:
# ─── Custom Layer 1: StripMask ───────────────────────────────
@tf.keras.utils.register_keras_serializable()
class StripMask(tf.keras.layers.Layer):
    """Menghapus mask agar kompatibel dengan GlobalMaxPooling1D."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True

    def call(self, inputs):
        return inputs

    def compute_mask(self, inputs, mask=None):
        return None

    def get_config(self):
        return super().get_config()


# ─── Custom Layer 2: AttentionPooling ────────────
@tf.keras.utils.register_keras_serializable()
class AttentionPooling(tf.keras.layers.Layer):
    """
    Self-attention pooling: memberikan bobot lebih
    pada token yang paling relevan untuk klasifikasi.
    
    Perbaikan v3: mask selalu di-handle dengan aman
    (tidak error bila mask=None).
    """
    def __init__(self, units=128, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True
        self.units     = units
        self.attention = tf.keras.layers.Dense(units, activation='tanh')
        self.score     = tf.keras.layers.Dense(1)

    def call(self, inputs, mask=None, training=False):
        attn_weights = self.score(self.attention(inputs))

        if mask is not None:
            mask_f = tf.cast(tf.expand_dims(mask, -1), attn_weights.dtype)
            attn_weights = attn_weights + (1.0 - mask_f) * -1e9

        attn_weights = tf.nn.softmax(attn_weights, axis=1)
        return tf.reduce_sum(inputs * attn_weights, axis=1)

    def compute_mask(self, inputs, mask=None):
        return None

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units})
        return config


# ─── Custom Loss: Focal Loss + Label Smoothing ────────────────
@tf.keras.utils.register_keras_serializable()
class FocalLoss(tf.keras.losses.Loss):
    """
    Focal Loss (Lin et al., 2017) dengan label smoothing.
    Membantu model fokus pada sampel yang sulit diprediksi
    dan mengurangi efek kelas dominan.
    gamma=2.0, smoothing=0.05
    """
    def __init__(self, gamma=2.0, smoothing=0.05, name='focal_loss', **kwargs):
        super().__init__(name=name, reduction='none', **kwargs)
        self.gamma     = gamma
        self.smoothing = smoothing

    def call(self, y_true, y_pred):
        y_pred  = tf.cast(y_pred, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        n_cls   = tf.cast(tf.shape(y_pred)[-1], tf.float32)

        y_oh    = tf.one_hot(tf.cast(y_true, tf.int32), depth=tf.shape(y_pred)[-1])
        y_oh    = y_oh * (1.0 - self.smoothing) + (self.smoothing / n_cls)

        ce      = -tf.reduce_sum(y_oh * tf.math.log(y_pred), axis=-1)
        p_cor   = tf.reduce_sum(y_oh * y_pred, axis=-1)
        return tf.pow(1.0 - p_cor, self.gamma) * ce

    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'smoothing': self.smoothing})
        return config


# ─── Custom Callback: KAVA Trainer ───────────────────────────
class KAVACallback(tf.keras.callbacks.Callback):
    """
    Custom Callback yang menggabungkan:
    1. Early Stopping (patience-based)
    2. Model checkpoint (save best)
    3. TensorBoard custom scalar logging
    """
    def __init__(self, log_dir, patience=12, model_path='kava_best_model.keras'):
        super().__init__()
        self.log_dir      = log_dir
        self.patience     = patience
        self.model_path   = model_path
        self.best_val_acc = 0.0
        self.wait         = 0
        self.writer       = tf.summary.create_file_writer(log_dir)

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        val_acc = logs.get('val_accuracy', 0)

        # Tulis semua metrik ke TensorBoard
        with self.writer.as_default():
            for name, val in logs.items():
                tf.summary.scalar(name, data=val, step=epoch)
            self.writer.flush()

        if val_acc > self.best_val_acc:
            self.best_val_acc = val_acc
            self.wait         = 0
            self.model.save(self.model_path)
            print(f"  ✅ Epoch {epoch+1}: Tersimpan! val_acc={val_acc:.4f} ({val_acc*100:.2f}%)")
        else:
            self.wait += 1
            print(f"  ⏳ Epoch {epoch+1}: Tidak ada kenaikan ({self.wait}/{self.patience}) — best={self.best_val_acc*100:.2f}%")
            if self.wait >= self.patience:
                self.model.stop_training = True
                print(f"  🛑 Early Stopping! Best val_acc: {self.best_val_acc*100:.2f}%")

    def on_train_end(self, logs=None):
        print(f"\n{'='*60}")
        print(f"Training Selesai! Best Val Accuracy: {self.best_val_acc*100:.2f}%")
        print(f"Model disimpan di: {self.model_path}")
        self.writer.close()


print("✅ Semua komponen kustom siap (3 Custom Layer, 1 Custom Loss, 1 Custom Callback)")

✅ Semua komponen kustom siap (3 Custom Layer, 1 Custom Loss, 1 Custom Callback)


# Bangun Model

In [15]:
def build_kava_model(vocab_size, embed_dim, max_len, num_numeric, num_classes):
    """
    Arsitektur KAVA v3 — Dual-Input Deep Learning:

    Jalur Teks:
      Embedding(128) → SpatialDropout1D(0.3)
         ├─ AttentionPooling(128) [Custom Layer]
         └─ StripMask → BiLSTM(128) → GlobalMaxPool + GlobalAvgPool
      → Concatenate → BatchNorm → Dropout(0.4)

    Jalur Numerik:
      Dense(64, relu) → BatchNorm → Dropout(0.2)

    Fusion:
      Concatenate → Dense(512, relu) → BatchNorm → Dropout(0.4)
                 → Dense(256, relu) → BatchNorm → Dropout(0.3)
                 → Dense(num_classes, softmax)
    """
    # ── Input 1: Teks CV ──────────────────────────────────
    text_input = keras.Input(shape=(max_len,), name='text_input', dtype='int32')
    emb = layers.Embedding(
        vocab_size, embed_dim,
        mask_zero=True, name='embedding'
    )(text_input)
    emb = layers.SpatialDropout1D(0.3)(emb)
    # Jalur A: Custom Attention Pooling
    x_attn = AttentionPooling(units=embed_dim, name='attention_pool')(emb)

    # Jalur B: BiLSTM 
    x_unmasked = StripMask(name='strip_mask')(emb)
    x_lstm = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1),
        name='bilstm_1'
    )(x_unmasked)
    x_max = layers.GlobalMaxPooling1D(name='global_max')(x_lstm)
    x_avg = layers.GlobalAveragePooling1D(name='global_avg')(x_lstm)

    # Gabungkan semua jalur teks
    x_text = layers.Concatenate(name='text_concat')([x_attn, x_max, x_avg])
    x_text = layers.BatchNormalization()(x_text)
    x_text = layers.Dropout(0.4)(x_text)

    # ── Input 2: Fitur Numerik ────────────────────────────
    numeric_input = keras.Input(shape=(num_numeric,), name='numeric_input', dtype='float32')
    x_num = layers.Dense(64, activation='relu', name='numeric_dense')(numeric_input)
    x_num = layers.BatchNormalization()(x_num)
    x_num = layers.Dropout(0.2)(x_num)

    # ── Fusion ────────────────────────────────────────────
    combined = layers.Concatenate(name='fusion_concat')([x_text, x_num])
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(2e-4),
                     name='dense_512')(combined)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(2e-4),
                     name='dense_256')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    output = layers.Dense(
        num_classes, activation='softmax',
        dtype='float32', name='output'
    )(x)

    return keras.Model(
        inputs=[text_input, numeric_input],
        outputs=output,
        name='KAVA_v3'
    )


model = build_kava_model(
    vocab_size   = vocab_size,
    embed_dim    = EMBED_DIM,
    max_len      = MAX_LEN,
    num_numeric  = NUM_NUMERIC_FEATURES,
    num_classes  = num_classes
)

model.summary()
print(f"\nTotal trainable params: {model.count_params():,}")

Model: "KAVA_v3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 text_input (InputLayer)        [(None, 300)]        0           []                               
                                                                                                  
 embedding (Embedding)          (None, 300, 128)     2560000     ['text_input[0][0]']             
                                                                                                  
 spatial_dropout1d (SpatialDrop  (None, 300, 128)    0           ['embedding[0][0]']              
 out1D)                                                                                           
                                                                                                  
 strip_mask (StripMask)         (None, 300, 128)     0           ['spatial_dropout1d[0][0]']

# Training Model

In [16]:
EPOCHS = 60

total_steps   = EPOCHS * len(train_ds)
warmup_steps  = int(total_steps * 0.08)


class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """
    Linear warmup + Cosine decay LR schedule.
    Mencegah model 'meledak' di awal training.
    peak_lr=1e-3 (↓ dari 2e-3 yang terlalu besar)
    """
    def __init__(self, peak_lr, total_steps, warmup_steps):
        self.peak_lr      = peak_lr
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)

    def __call__(self, step):
        step       = tf.cast(step, tf.float32)
        warmup_lr  = self.peak_lr * (step / self.warmup_steps)

        cosine_steps = tf.maximum(step - self.warmup_steps, 0.0)
        cosine_total = self.total_steps - self.warmup_steps
        cosine_lr    = self.peak_lr * 0.5 * (1 + tf.cos(np.pi * cosine_steps / cosine_total))

        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'peak_lr':      self.peak_lr,
            'total_steps':  int(self.total_steps.numpy()),
            'warmup_steps': int(self.warmup_steps.numpy())
        }


lr_schedule = WarmupCosineDecay(peak_lr=1e-3, total_steps=total_steps, warmup_steps=warmup_steps)
optimizer   = tf.keras.optimizers.Adam(
    learning_rate=lr_schedule,
    clipnorm=1.0
)
loss_fn     = FocalLoss(gamma=2.0, smoothing=0.05)

# Metrik
train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
val_acc_metric   = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
train_mae_metric = tf.keras.metrics.MeanAbsoluteError(name='train_mae')
val_mae_metric   = tf.keras.metrics.MeanAbsoluteError(name='val_mae')

print(f"EPOCHS={EPOCHS} | peak_lr=1e-3 | warmup={warmup_steps} steps")

EPOCHS=60 | peak_lr=1e-3 | warmup=672 steps


In [17]:
@tf.function
def train_step(text_batch, num_batch, y_batch, sample_weights):
    with tf.GradientTape() as tape:
        preds           = model({'text_input': text_batch, 'numeric_input': num_batch},
                                training=True)
        per_sample_loss = loss_fn(y_batch, preds)
        loss_value      = tf.reduce_mean(per_sample_loss * sample_weights)

    grads = tape.gradient(loss_value, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    train_acc_metric.update_state(y_batch, preds)
    y_oh = tf.one_hot(tf.cast(y_batch, tf.int32), depth=num_classes)
    train_mae_metric.update_state(y_oh, preds)
    return loss_value


@tf.function
def val_step(text_batch, num_batch, y_batch):
    preds  = model({'text_input': text_batch, 'numeric_input': num_batch},
                   training=False)
    v_loss = tf.reduce_mean(loss_fn(y_batch, preds))
    val_acc_metric.update_state(y_batch, preds)
    y_oh = tf.one_hot(tf.cast(y_batch, tf.int32), depth=num_classes)
    val_mae_metric.update_state(y_oh, preds)
    return v_loss


print("✅ train_step & val_step (tf.GradientTape) siap")

✅ train_step & val_step (tf.GradientTape) siap


In [18]:
LOG_DIR    = f"logs/kava/{datetime.now().strftime('%Y%m%d-%H%M%S')}"
MODEL_PATH = 'kava_best_model.keras'
os.makedirs(LOG_DIR, exist_ok=True)

train_writer = tf.summary.create_file_writer(f"{LOG_DIR}/train")
val_writer   = tf.summary.create_file_writer(f"{LOG_DIR}/val")
print(f"TensorBoard: tensorboard --logdir=logs/kava")
print(f"Log dir    : {LOG_DIR}")
print()

TensorBoard: tensorboard --logdir=logs/kava
Log dir    : logs/kava/20260520-112643



In [19]:
kava_cb = KAVACallback(log_dir=f"{LOG_DIR}/callback",
                       patience=12, model_path=MODEL_PATH)
kava_cb.set_model(model)

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'train_mae':  [], 'val_mae':  []
}

for epoch in range(EPOCHS):
    start = time.time()

    # ─── Training pass ───────────────────────────────────
    total_tloss, n_train = 0.0, 0
    for (x_batch, y_batch, w_batch) in train_ds:
        loss = train_step(
            x_batch['text_input'], x_batch['numeric_input'],
            y_batch, w_batch
        )
        total_tloss += loss; n_train += 1

    t_loss = total_tloss / n_train
    t_acc  = train_acc_metric.result()
    t_mae  = train_mae_metric.result()

    # ─── Validation pass ─────────────────────────────────
    total_vloss, n_val = 0.0, 0
    for (x_batch, y_batch) in val_ds:
        vloss = val_step(x_batch['text_input'], x_batch['numeric_input'], y_batch)
        total_vloss += vloss; n_val += 1

    v_loss = total_vloss / n_val
    v_acc  = val_acc_metric.result()
    v_mae  = val_mae_metric.result()

    elapsed = time.time() - start

    # TensorBoard logging
    with train_writer.as_default():
        tf.summary.scalar('loss', t_loss, step=epoch)
        tf.summary.scalar('accuracy', float(t_acc), step=epoch)
        tf.summary.scalar('mae', float(t_mae), step=epoch)
    with val_writer.as_default():
        tf.summary.scalar('loss', v_loss, step=epoch)
        tf.summary.scalar('accuracy', float(v_acc), step=epoch)
        tf.summary.scalar('mae', float(v_mae), step=epoch)

    # Simpan history
    history['train_loss'].append(float(t_loss))
    history['val_loss'].append(float(v_loss))
    history['train_acc'].append(float(t_acc))
    history['val_acc'].append(float(v_acc))
    history['train_mae'].append(float(t_mae))
    history['val_mae'].append(float(v_mae))

    # Reset metrik
    train_acc_metric.reset_state()
    val_acc_metric.reset_state()
    train_mae_metric.reset_state()
    val_mae_metric.reset_state()

    print(f"Epoch {epoch+1:>3}/{EPOCHS} | "
          f"loss={float(t_loss):.4f} acc={float(t_acc)*100:.2f}% mae={float(t_mae):.4f} | "
          f"val_loss={float(v_loss):.4f} val_acc={float(v_acc)*100:.2f}% val_mae={float(v_mae):.4f} | "
          f"{elapsed:.1f}s")

    # KAVACallback
    kava_cb.on_epoch_end(epoch, logs={
        'loss': float(t_loss), 'accuracy': float(t_acc), 'mae': float(t_mae),
        'val_loss': float(v_loss), 'val_accuracy': float(v_acc), 'val_mae': float(v_mae)
    })

    if model.stop_training:
        break

kava_cb.on_train_end()

Epoch   1/60 | loss=3.7801 acc=5.20% mae=0.0947 | val_loss=2.7686 val_acc=3.76% val_mae=0.0949 | 317.1s
  ✅ Epoch 1: Tersimpan! val_acc=0.0376 (3.76%)
Epoch   2/60 | loss=3.1230 acc=12.62% mae=0.0905 | val_loss=2.6557 val_acc=9.41% val_mae=0.0940 | 310.2s
  ✅ Epoch 2: Tersimpan! val_acc=0.0941 (9.41%)
Epoch   3/60 | loss=2.1973 acc=32.44% mae=0.0766 | val_loss=2.0746 val_acc=25.46% val_mae=0.0877 | 309.6s
  ✅ Epoch 3: Tersimpan! val_acc=0.2546 (25.46%)
Epoch   4/60 | loss=1.4454 acc=50.97% mae=0.0611 | val_loss=1.6500 val_acc=40.46% val_mae=0.0743 | 296.9s
  ✅ Epoch 4: Tersimpan! val_acc=0.4046 (40.46%)
Epoch   5/60 | loss=1.0384 acc=62.53% mae=0.0504 | val_loss=1.2892 val_acc=58.86% val_mae=0.0526 | 311.7s
  ✅ Epoch 5: Tersimpan! val_acc=0.5886 (58.86%)
Epoch   6/60 | loss=0.7643 acc=70.43% mae=0.0426 | val_loss=1.1105 val_acc=64.82% val_mae=0.0437 | 310.8s
  ✅ Epoch 6: Tersimpan! val_acc=0.6482 (64.82%)
Epoch   7/60 | loss=0.5410 acc=76.89% mae=0.0354 | val_loss=1.1158 val_acc=67.64%

# Evaluasi

In [20]:
best_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        'StripMask':        StripMask,
        'AttentionPooling': AttentionPooling,
        'FocalLoss':        FocalLoss
    }
)

y_pred_prob = best_model.predict(
    {'text_input': vectorizer(X_text_test),
     'numeric_input': X_num_test_scaled},
    batch_size=128, verbose=1
)
y_pred = np.argmax(y_pred_prob, axis=1)

acc      = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
y_test_oh = tf.one_hot(y_test, depth=num_classes).numpy()
final_mae = float(np.mean(np.abs(y_pred_prob - y_test_oh)))

print("=" * 60)
print(f"TEST ACCURACY : {acc*100:.2f}%  (target: ≥85%)")
print(f"MACRO F1      : {macro_f1*100:.2f}%")
print(f"MAE           : {final_mae:.5f}  (target: ≤0.02)")
print("=" * 60)

if acc >= 0.85:
    print("✅ MVP TERCAPAI: Akurasi ≥ 85%")
else:
    print(f"⚠️  Akurasi {acc*100:.2f}% — belum 85%. Coba jalankan ulang atau tambah epochs.")

if final_mae <= 0.02:
    print("✅ MVP TERCAPAI: MAE ≤ 0.02")
else:
    print(f"⚠️  MAE {final_mae:.5f} — belum 0.02")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))

15/15 [==============================] - 8s 431ms/step
TEST ACCURACY : 71.73%  (target: ≥85%)
MACRO F1      : 68.97%
MAE           : 0.03055  (target: ≤0.02)
⚠️  Akurasi 71.73% — belum 85%. Coba jalankan ulang atau tambah epochs.
⚠️  MAE 0.03055 — belum 0.02

Classification Report:
                              precision    recall  f1-score   support

                 Agriculture      0.864     0.576     0.691        33
       Architecture & Design      0.609     0.368     0.459        38
      Arts & Creative Design      0.646     0.708     0.675       144
            Business Analyst      0.519     0.702     0.597        57
                Construction      0.714     0.565     0.631        62
  Data Science & Engineering      0.796     0.880     0.836       133
        Education & Training      0.768     0.683     0.723        63
     Engineering & Technical      0.744     0.841     0.790       145
        Finance & Accounting      0.778     0.826     0.801       178
          Food &

# Visualisasi Learning Curve

In [22]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_range = range(1, len(history['train_acc']) + 1)

axes[0].plot(epochs_range, history['train_acc'], label='Train')
axes[0].plot(epochs_range, history['val_acc'],   label='Val')
axes[0].axhline(0.85, color='r', linestyle='--', label='Target 85%')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_range, history['train_loss'], label='Train')
axes[1].plot(epochs_range, history['val_loss'],   label='Val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs_range, history['train_mae'], label='Train')
axes[2].plot(epochs_range, history['val_mae'],   label='Val')
axes[2].axhline(0.02, color='r', linestyle='--', label='Target MAE 0.02')
axes[2].set_title('MAE'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('kava_learning_curve.png', dpi=150, bbox_inches='tight')
print("Grafik disimpan: kava_learning_curve.png")

Grafik disimpan: kava_learning_curve.png


# Pipeline Keyword Matching

In [25]:
KEYWORD_DOMAIN_MAP = {
    'Software & Web Development':    ['python','javascript','react','node','java','c++','sql','git',
                                       'docker','kubernetes','api','testing','devops','ci/cd','agile'],
    'Data Science & Engineering':    ['machine learning','deep learning','tensorflow','pytorch',
                                       'pandas','numpy','sql','etl','data pipeline','statistics',
                                       'spark','hadoop','visualization','nlp','scikit-learn'],
    'IT Infrastructure & Security':  ['network','firewall','vpn','linux','aws','azure','gcp',
                                       'cloud','sap','cybersecurity','active directory','vmware'],
    'Finance & Accounting':          ['accounting','finance','excel','sap','audit','tax',
                                       'budget','financial analysis','gaap','quickbooks','erp'],
    'Human Resources':               ['recruitment','onboarding','hr','payroll','performance',
                                       'training','hris','labor law','talent acquisition'],
    'Sales & Business Development':  ['sales','crm','negotiation','b2b','revenue','pipeline',
                                       'account management','lead generation','customer success'],
    'Healthcare & Wellness':         ['patient care','clinical','medical','nursing','diagnosis',
                                       'ehr','healthcare','pharmacology','therapy','anatomy'],
    'Arts & Creative Design':        ['photoshop','illustrator','figma','ui','ux','graphic design',
                                       'typography','branding','animation','video editing'],
    'Education & Training':          ['teaching','curriculum','lesson plan','assessment',
                                       'e-learning','classroom management','mentoring','coaching'],
    'Engineering & Technical':       ['autocad','solidworks','engineering','mechanical','electrical',
                                       'civil','project management','cad','simulation'],
    'Construction':                  ['construction','project management','blueprints','safety',
                                       'estimating','scheduling','site management'],
    'Transportation & Logistics':    ['logistics','supply chain','inventory','warehouse',
                                       'fleet management','shipping','procurement'],
    'Food & Hospitality':            ['food safety','menu','customer service','hospitality',
                                       'cooking','catering','restaurant management'],
    'Legal & Advocacy':              ['legal','litigation','contract','compliance','regulatory',
                                       'research','advocacy','corporate law'],
    'PR & Digital Media':            ['social media','content','seo','marketing','pr',
                                       'communication','brand','campaign','analytics'],
    'Business Analyst':              ['requirements','business analysis','sql','process improvement',
                                       'stakeholder','documentation','agile','jira'],
    'Management Consulting':         ['strategy','consulting','business process','change management',
                                       'stakeholder','presentation','analysis'],
    'Management & Operations':       ['leadership','operations','strategic planning','team management',
                                       'kpi','budget','performance','pmo'],
    'Agriculture':                   ['farming','crop','soil','irrigation','agronomy','pest','harvest'],
    'Architecture & Design':         ['architecture','revit','autocad','sketching','structural',
                                       'interior design','building code','3d modeling']
}


def keyword_matching_pipeline(cv_text: str) -> dict:
    cv_lower = cv_text.lower()
    scores   = {}
    for domain, keywords in KEYWORD_DOMAIN_MAP.items():
        matched      = [kw for kw in keywords if kw in cv_lower]
        scores[domain] = round(len(matched) / len(keywords) * 100, 1) if keywords else 0.0
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))


def get_keyword_top3(cv_text: str) -> list:
    scores = keyword_matching_pipeline(cv_text)
    top3   = list(scores.items())[:3]
    return [{'rank': i+1, 'role': role, 'keyword_score': score}
            for i, (role, score) in enumerate(top3)]


# Uji coba
sample_cv = "python machine learning tensorflow pandas sql data visualization statistics"
kw_result = keyword_matching_pipeline(clean_text(sample_cv))
print("=== KEYWORD MATCHING TOP-3 ===")
for domain, score in list(kw_result.items())[:3]:
    print(f"  {domain:40s} {score:.1f}%")

=== KEYWORD MATCHING TOP-3 ===
  Data Science & Engineering               40.0%
  Software & Web Development               13.3%
  Business Analyst                         12.5%


# Inference: Prediksi Top-3 Role

In [26]:
def predict_top3_cv(text: str, exp_years: float, certs_count: int,
                    has_edu: int, skills_raw: str = "") -> list:
    """
    Prediksi Top-3 Career Role Match.
    Skor final = 80% DL confidence + 20% keyword matching.
    """
    is_fresh = 1 if exp_years < 2 else 0
    is_exp   = 1 if exp_years >= 5 else 0
    has_cert = 1 if certs_count > 0 else 0

    skills_count_est = min(len(skills_raw.split(',')) if skills_raw else 15, 36)

    num_feat   = np.array([[exp_years, skills_count_est, has_cert, certs_count,
                            has_edu, is_fresh, is_exp, 1]], dtype=np.float32)
    num_scaled = scaler.transform(num_feat)

    cleaned    = clean_text(text)
    text_token = vectorizer(np.array([cleaned]))
    probs      = best_model.predict(
        {'text_input': text_token, 'numeric_input': num_scaled},
        verbose=0
    )[0]

    kw_scores = keyword_matching_pipeline(cleaned)

    combined_scores = {}
    for i, role in enumerate(le.classes_):
        dl_score  = float(probs[i]) * 100
        kw_score  = kw_scores.get(role, 0.0)
        combined_scores[role] = {
            'dl_confidence': round(dl_score, 2),
            'keyword_score': kw_score,
            'final_score':   round(dl_score * 0.8 + kw_score * 0.2, 2)
        }

    top3 = sorted(combined_scores.items(),
                  key=lambda x: x[1]['final_score'], reverse=True)[:3]
    return [{'rank': i+1, 'role': role, **scores}
            for i, (role, scores) in enumerate(top3)]


# Uji coba
sample_text = """Experienced data scientist with Python, TensorFlow, scikit-learn,
pandas, numpy. Built ML models and recommendation systems. AWS and Docker."""

top3 = predict_top3_cv(
    text        = sample_text,
    exp_years   = 3.0,
    certs_count = 2,
    has_edu     = 1,
    skills_raw  = "python, tensorflow, pandas, sql, aws"
)

print("=== TOP-3 ROLE MATCHES ===")
for r in top3:
    print(f"  #{r['rank']} {r['role']:35s} DL:{r['dl_confidence']:.1f}% "
          f"KW:{r['keyword_score']:.1f}% Final:{r['final_score']:.1f}%")

=== TOP-3 ROLE MATCHES ===
  #1 Data Science & Engineering          DL:100.0% KW:26.7% Final:85.3%
  #2 Software & Web Development          DL:0.0% KW:13.3% Final:2.7%
  #3 Arts & Creative Design              DL:0.0% KW:10.0% Final:2.0%


# Skill Gap Database & Analysis

In [27]:
def build_skill_gap_database(df: pd.DataFrame) -> dict:
    """Bangun database top-20 skill per role dari data training."""
    skill_db = {}
    for group in df['category_grouped'].unique():
        group_df   = df[df['category_grouped'] == group]
        all_skills = []
        for raw in group_df['skills'].dropna():
            for s in str(raw).split(','):
                s = s.strip().lower()
                if len(s) > 2:
                    all_skills.append(s)
        freq  = Counter(all_skills)
        top20 = [skill for skill, cnt in freq.most_common(30) if cnt >= 2][:20]
        skill_db[group] = top20
    return skill_db


skill_db = build_skill_gap_database(df)

with open('skill_gap_db.json', 'w') as f:
    json.dump(skill_db, f, indent=2)
print("✅ skill_gap_db.json disimpan")


def analyze_skill_gap(cv_skills_raw: str, predicted_role: str) -> dict:
    """
    Bandingkan skills CV dengan requirements role.
    Return: matched, missing, coverage_pct (untuk progress bar frontend).
    """
    cv_skills = set(s.strip().lower() for s in cv_skills_raw.split(',') if s.strip())
    required  = skill_db.get(predicted_role, [])
    if not required:
        return {'error': f'Role tidak ditemukan: {predicted_role}'}

    matched  = [s for s in required if any(s in cs or cs in s for cs in cv_skills)]
    missing  = [s for s in required if s not in matched]
    coverage = round(len(matched) / len(required) * 100, 1) if required else 0

    return {
        'predicted_role':     predicted_role,
        'required_skills':    required,
        'matched_skills':     matched,
        'missing_skills':     missing[:10],
        'coverage_pct':       coverage,
        'progress_bar_value': round(coverage / 100, 3)
    }


# Uji coba
cv_skills_input = "python, machine learning, tensorflow, pandas, sql, data visualization"
print("\n=== SKILL GAP ANALYSIS ===")
for r in top3:
    gap = analyze_skill_gap(cv_skills_input, r['role'])
    print(f"\n#{r['rank']} {r['role']} ({r['final_score']:.1f}%)")
    print(f"   ✅ Matched  : {gap.get('matched_skills', [])}")
    print(f"   ❌ Missing  : {gap.get('missing_skills', [])}")
    print(f"   📊 Coverage : {gap.get('coverage_pct', 0)}%")

✅ skill_gap_db.json disimpan

=== SKILL GAP ANALYSIS ===

#1 Data Science & Engineering (85.3%)
   ✅ Matched  : ['sql', 'pl/sql', 'python', 't-sql', 'sql server', 'mysql']
   ❌ Missing  : ['oracle', 'java', 'ssis', 'c++', 'html', 'unix', 'ssrs', 'xml', 'tableau', 'etl']
   📊 Coverage : 30.0%

#2 Software & Web Development (2.7%)
   ✅ Matched  : ['sql', 'mysql', 'python']
   ❌ Missing  : ['javascript', 'jquery', 'html', 'css', 'java', 'git', 'xml', 'ajax', 'jenkins', 'html5']
   📊 Coverage : 15.0%

#3 Arts & Creative Design (2.0%)
   ✅ Matched  : []
   ❌ Missing  : ['customer service', 'sales', 'excel', 'photoshop', 'illustrator', 'time management', 'microsoft office', 'word', 'adobe photoshop', 'project management']
   📊 Coverage : 0.0%


# AI Career Coach (Mistral API)

In [31]:
def get_ai_career_coach(top1_role: str, exp_years: float,
                        skills_raw: str, missing_skills: list) -> str:
    """
    AI Career Coach via Mistral REST API langsung (pakai requests).
    Tidak bergantung pada versi library mistralai sama sekali.
    """
    MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY", "")
    if not MISTRAL_API_KEY:
        return "⚠️  Belum set API key."

    prompt = f"""Anda adalah KAVA (Karier AI Validasi Asisten), asisten karir profesional.

Profil Kandidat:
- Role diprediksi: {top1_role}
- Pengalaman: {exp_years} tahun
- Skills yang dimiliki: {skills_raw}
- Skills yang perlu ditingkatkan: {', '.join(missing_skills)}

Berikan dalam Bahasa Indonesia (maksimal 200 kata):
1. Saran Karir: Langkah konkret berikutnya
2. Skill Prioritas: 3 missing skill terpenting dan alasannya
3. Rekomendasi Sertifikasi: 1 sertifikasi internasional paling relevan"""

    try:
        response = requests.post(
            "https://api.mistral.ai/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {MISTRAL_API_KEY}",
                "Content-Type": "application/json"
            },
            json={
                "model": "mistral-small-latest",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 400
            },
            timeout=30
        )
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]

    except requests.exceptions.HTTPError as e:
        return f"❌ HTTP Error {response.status_code}: {response.text}"
    except Exception as e:
        return f"❌ Error: {str(e)}"


os.environ["MISTRAL_API_KEY"] = "PbaoLwm4UiuY6BlOqTOCymP18nvuG9bf"

# Uji coba
if top3:
    gap_top1 = analyze_skill_gap(cv_skills_input, top3[0]['role'])
    coach    = get_ai_career_coach(
        top1_role      = top3[0]['role'],
        exp_years      = 3.0,
        skills_raw     = cv_skills_input,
        missing_skills = gap_top1.get('missing_skills', [])
    )
    print("=== AI CAREER COACH (Mistral) ===")
    print(coach)

=== AI CAREER COACH (Mistral) ===
### **Saran Karir untuk Kandidat Data Science & Engineering**
Dengan pengalaman 3 tahun di bidang Data Science & Engineering, langkah konkret berikutnya adalah:
1. **Spesialisasi Role**: Fokus pada *Data Engineering* (ETL, pipeline, database) atau *Machine Learning Engineering* (MLOps, deployment) untuk meningkatkan daya saing.
2. **Proyek Nyata**: Bangun portofolio dengan proyek yang melibatkan *ETL* (misal: SSIS) dan *ML deployment* (misal: TensorFlow Serving) untuk menunjukkan kemampuan praktis.
3. **Networking**: Ikuti komunitas data (seperti Kaggle, DataTalks.Club) atau LinkedIn untuk peluang kerja dan kolaborasi.

### **3 Skill Prioritas untuk Ditingkatkan**
1. **SQL (Oracle)** – *Kunci untuk data engineering*: Oracle SQL sangat dibutuhkan di perusahaan besar (perbankan, telekomunikasi).
2. **ETL (SSIS)** – *Standar industri*: Penting untuk integrasi data dan pipeline, terutama di perusahaan dengan sistem legacy.
3. **Tableau** – *Visualisasi dat

# Simpan Semua Artifacts

In [35]:
SAVE_DIR = "./artifacts/"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Menyimpan ke: {SAVE_DIR}")

# 1. Model — format .keras (TF production-ready)
model_path = f"{SAVE_DIR}kava_best_model.keras"
best_model.save(model_path)

# 2. TextVectorization weights
vec_payload = {'config': vectorizer.get_config(), 'weights': vectorizer.get_weights()}
with open(f"{SAVE_DIR}vectorizer.pkl", "wb") as f:
    pickle.dump(vec_payload, f)

# 3. StandardScaler
with open(f"{SAVE_DIR}scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# 4. LabelEncoder
with open(f"{SAVE_DIR}label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

# 5. Skill Gap DB
with open(f"{SAVE_DIR}skill_gap_db.json", "w") as f:
    json.dump(skill_db, f, indent=2)

# 6. Keyword Domain Map
with open(f"{SAVE_DIR}keyword_domain_map.json", "w") as f:
    json.dump(KEYWORD_DOMAIN_MAP, f, indent=2)

# 7. Model Metadata (untuk Frontend/Backend)
metadata = {
    'model_name':           'KAVA v3.0',
    'num_classes':          num_classes,
    'class_names':          le.classes_.tolist(),
    'max_len':              MAX_LEN,
    'embed_dim':            EMBED_DIM,
    'max_tokens':           MAX_TOKENS,
    'num_numeric_features': NUM_NUMERIC_FEATURES,
    'numeric_columns':      NUMERIC_COLS,
    'architecture':         'BiLSTM(128) + AttentionPooling(128) + Numeric Fusion',
    'test_accuracy':        round(float(acc), 4),
    'macro_f1':             round(float(macro_f1), 4),
    'mae':                  round(float(final_mae), 5),
    'trained_at':           datetime.now().isoformat()
}
with open(f"{SAVE_DIR}model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# 8. Training history
with open(f"{SAVE_DIR}training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\n✅ Artifact yang dihasilkan:")
artifacts = [
    'kava_best_model.keras', 'vectorizer.pkl', 'scaler.pkl',
    'label_encoder.pkl', 'skill_gap_db.json', 'keyword_domain_map.json',
    'model_metadata.json', 'training_history.json'
]
for fname in artifacts:
    fpath = f"{SAVE_DIR}{fname}"
    size  = os.path.getsize(fpath) / 1024 if os.path.exists(fpath) else 0
    print(f"   {fname:40s} {size:8.1f} KB")

Menyimpan ke: ./artifacts/

✅ Artifact yang dihasilkan:
   kava_best_model.keras                     13129.6 KB
   vectorizer.pkl                              283.3 KB
   scaler.pkl                                    0.6 KB
   label_encoder.pkl                             0.7 KB
   skill_gap_db.json                             8.3 KB
   keyword_domain_map.json                       3.9 KB
   model_metadata.json                           1.1 KB
   training_history.json                         6.7 KB


# End-to-End Test + Format Response untuk Fullstack

In [36]:
sample_payload = {
    "text":             "Experienced data scientist with Python, TensorFlow, scikit-learn. "
                        "Built ML models, recommendation systems, and NLP pipelines. AWS & Docker.",
    "skills_raw":       "python, tensorflow, scikit-learn, nlp, aws, docker, sql, pandas",
    "experience_years": 3.0,
    "cert_count":       2,
    "has_education":    1
}

top3_result = predict_top3_cv(
    text        = sample_payload["text"],
    exp_years   = sample_payload["experience_years"],
    certs_count = sample_payload["cert_count"],
    has_edu     = sample_payload["has_education"],
    skills_raw  = sample_payload["skills_raw"]
)
gap_result = analyze_skill_gap(sample_payload["skills_raw"], top3_result[0]["role"])

full_response = {
    "status":             "success",
    "top3_role_matches":  top3_result,
    "skill_gap":          gap_result,
    "ai_career_coach":    "(teks dari Mistral API — butuh API key saat deploy)"
}

print("=== RESPONSE JSON UNTUK TIM FULLSTACK ===")
print(json.dumps(full_response, indent=2, ensure_ascii=False))

=== RESPONSE JSON UNTUK TIM FULLSTACK ===
{
  "status": "success",
  "top3_role_matches": [
    {
      "rank": 1,
      "role": "Data Science & Engineering",
      "dl_confidence": 99.96,
      "keyword_score": 20.0,
      "final_score": 83.97
    },
    {
      "rank": 2,
      "role": "Software & Web Development",
      "dl_confidence": 0.03,
      "keyword_score": 13.3,
      "final_score": 2.69
    },
    {
      "rank": 3,
      "role": "Sales & Business Development",
      "dl_confidence": 0.0,
      "keyword_score": 11.1,
      "final_score": 2.22
    }
  ],
  "skill_gap": {
    "predicted_role": "Data Science & Engineering",
    "required_skills": [
      "sql",
      "pl/sql",
      "oracle",
      "python",
      "java",
      "t-sql",
      "ssis",
      "sql server",
      "c++",
      "html",
      "unix",
      "ssrs",
      "xml",
      "mysql",
      "tableau",
      "etl",
      "ms access",
      "linux",
      "javascript",
      "excel"
    ],
    "matched_skills":